# 3. Predict — simulated CRM data (`data/crm_simulated/`)

Same steps as `python -m src.predict --output reports/crm/simulated/final`, using the functions in `src/predict.py`.
Uses the frozen calibrated model saved by `1_train.ipynb`. Open deals are never labelled Lost.

Scores on simulated deals illustrate the workflow only.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file())
sys.path.insert(0, str(ROOT))

import tempfile
import pandas as pd
from src import predict, train
from src.outputs import Outputs


## Settings

Must match the settings used in `1_train.ipynb`.

In [2]:
DATA_DIR = ROOT / "data/crm_simulated"   # 100,000 simulated deals
QUICK = False            # True: tiny budgets, smoke test only (writes a *smoke* folder)
OUTPUT = train.default_output(DATA_DIR, QUICK)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT)

Data:   /Users/thaiphan/sources/miu/cs582-project/data/crm_simulated
Output: /Users/thaiphan/sources/miu/cs582-project/reports/crm/simulated/final


## 1. Load the scoring bundle

In [3]:
out = Outputs(OUTPUT)
scoring = predict.load(out)
print("Model:", scoring["model"].name)
print("Decision threshold (calibrated):", round(scoring["decision_threshold"], 4))
print("Features:", len(scoring["features"]), "| data:", out.source_data())

Model: tabnet_calibrated
Decision threshold (calibrated): 0.4872
Features: 18 | data: /Users/thaiphan/sources/miu/cs582-project/data/crm_simulated


## 2. Score the open Engaging deals

Writes `predictions/open_deal_predictions.csv`: P(Won), P(Lost), predicted class, a heuristic priority band,
the inputs that raise or lower each score, and a warning when the account is missing.

In [4]:
scores = predict.run(OUTPUT)
pd.Series({
    "scored deals": len(scores),
    "predicted Won": int(scores.predicted_outcome.eq("Won").sum()),
    "High / Medium / Low": "/".join(str(int(scores.priority.eq(p).sum())) for p in ["High", "Medium", "Low"]),
    "missing account (warning)": int(scores.account_missing.sum()),
    "mean P(Won)": round(scores.win_probability.mean(), 4),
})

PREDICTED: 22,810 open deals -> /Users/thaiphan/sources/miu/cs582-project/reports/crm/simulated/final/predictions/open_deal_predictions.csv


scored deals                           22810
predicted Won                          17146
High / Medium / Low          12192/7043/3575
missing account (warning)              15582
mean P(Won)                           0.6737
dtype: object

In [5]:
scores[["opportunity_id", "product", "sales_agent", "win_probability", "priority",
        "positive_factors", "negative_factors", "input_warning"]].head(10)

,opportunity_id,product,sales_agent,win_probability,priority,positive_factors,negative_factors,input_warning
0,C1A79556,GTS Enterprise,Vicki Laflamme,0.991489,High,product=GTS Enterprise (delta P(Won)=+0.1459);...,No negative reference sensitivity,
1,628B779C,GTS Enterprise,Hayden Neloms,0.991489,High,product=GTS Enterprise (delta P(Won)=+0.1585);...,No negative reference sensitivity,
2,73DA2D31,GTS Enterprise,Vicki Laflamme,0.991489,High,product=GTS Enterprise (delta P(Won)=+0.1453);...,No negative reference sensitivity,
3,DAD90E96,GTS Enterprise,Markita Hansen,0.990734,High,product=GTS Enterprise (delta P(Won)=+0.1606);...,sales_agent=Markita Hansen (delta P(Won)=-0.00...,
4,997AE6F3,GTS Enterprise,Markita Hansen,0.990491,High,product=GTS Enterprise (delta P(Won)=+0.0671);...,No negative reference sensitivity,
5,02674BA4,GTS Enterprise,Vicki Laflamme,0.990491,High,product=GTS Enterprise (delta P(Won)=+0.0653);...,No negative reference sensitivity,
6,734B7624,GTS Enterprise,Vicki Laflamme,0.990491,High,product=GTS Enterprise (delta P(Won)=+0.0646);...,No negative reference sensitivity,
7,EBCE721B,GTS Enterprise,Elease Gluck,0.990491,High,product=GTS Enterprise (delta P(Won)=+0.0698);...,No negative reference sensitivity,
8,81DD57F1,GTS Enterprise,Elease Gluck,0.990491,High,product=GTS Enterprise (delta P(Won)=+0.0704);...,No negative reference sensitivity,
9,BDC52897,GTS Enterprise,Markita Hansen,0.990491,High,product=GTS Enterprise (delta P(Won)=+0.0660);...,No negative reference sensitivity,


## 3. Score new deals from a CSV

New deals use the `sales_pipeline.csv` format: `opportunity_id, sales_agent, product, account, engage_date`.
This example takes five Engaging deals from the data as stand-ins; replace `new_deals` with your own file.

In [6]:
raw = pd.read_csv(out.source_data() / "sales_pipeline.csv")
new_deals = raw.loc[raw.deal_stage.eq("Engaging") & raw.account.notna(),
                    ["opportunity_id", "sales_agent", "product", "account", "engage_date"]].head(5)
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "new_deals.csv"
    new_deals.to_csv(path, index=False)
    new_scores = predict.score_file(path, OUTPUT, Path(folder) / "new_deal_scores.csv")
new_scores[["opportunity_id", "win_probability", "predicted_outcome", "priority", "positive_factors"]]

PREDICTED: 5 deals -> /var/folders/xb/q75wg0yn2gl2bysx0cgshv4w0000gn/T/tmpe21cu8nc/new_deal_scores.csv


,opportunity_id,win_probability,predicted_outcome,priority,positive_factors
0,9FF989FD,0.835722,Won,High,product=MG Lite (delta P(Won)=+0.0636); series...
1,DA15B925,0.797875,Won,High,sales_price=55 (delta P(Won)=+0.6653); series=...
2,A1B6D297,0.745954,Won,High,revenue=2403.58 (delta P(Won)=+0.1046); employ...
3,159D1384,0.430668,Lost,Medium,sales_price=5482 (delta P(Won)=+0.0062); engag...
4,F4EB9408,0.016904,Lost,Low,sector=medical (delta P(Won)=+0.0044); engage_...
